# Exploratory 50-record pilot
Run on an L4 or other native BF16 GPU. Source qualification remains pending. Publish the workflow branch before using the checkout below. No data or outputs are embedded in this notebook.

In [ ]:
import subprocess, sys
REF = "codex/50-sample-pilot"
subprocess.run(["git", "clone", "--branch", REF, "https://github.com/Dharshan2004/algoverse.git", "/content/algoverse-pilot-code"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "transformers==5.19.0", "mistral-common==1.12.0", "accelerate", "bitsandbytes", "huggingface_hub", "sentencepiece", "numpy"], check=True)


In [ ]:
from google.colab import files
from pathlib import Path
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False)
print(torch.cuda.get_device_name(0))
uploaded = files.upload()
assert len(uploaded) == 1, "Upload the original candidate_inputs.jsonl only"
source = Path('/content/candidate_inputs.jsonl')
source.write_bytes(next(iter(uploaded.values())))


In [ ]:
import subprocess, sys
repo = Path('/content/algoverse-pilot-code')
run = Path('/content/pilot-run')
subprocess.run([sys.executable, '-u', str(repo/'pilot.py'), '--input', str(source), '--out', str(run)], check=True)
for model in ['ministral','gemma']:
    subprocess.run([sys.executable, str(repo/'pilot_report.py'), str(run/model), '--input', str(source), '--check-states', '--out', str(run/'reports'/model)], check=True)
(run/'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
import shutil
files.download(shutil.make_archive(str(run), 'zip', run))
